# MLflow + Spark MLlib Demo

In [1]:
import os

import mlflow
import mlflow.spark
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import SparkSession


TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5001")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "spark-mllib-demo")

In [2]:
spark = (
    SparkSession.builder.appName("spark-mllib-mlflow-demo")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow tracking URI: {TRACKING_URI}")
print(f"Experiment: {EXPERIMENT_NAME}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 13:26:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


MLflow tracking URI: http://mlflow:5000
Experiment: spark-mllib-demo


In [3]:
train_rows = [
    (0.10, 1.10, 0.0),
    (0.20, 1.30, 0.0),
    (0.30, 1.70, 0.0),
    (0.35, 1.90, 0.0),
    (0.40, 2.20, 0.0),
    (0.55, 2.80, 0.0),
    (0.70, 3.10, 1.0),
    (0.80, 3.30, 1.0),
]
test_rows = [
    (0.90, 3.60, 1.0),
    (1.10, 4.00, 1.0),
    (0.25, 1.50, 0.0),
    (1.20, 4.30, 1.0),
]
columns = ["feature_1", "feature_2", "label"]

train_df = spark.createDataFrame(train_rows, columns)
test_df = spark.createDataFrame(test_rows, columns)

train_df.show()
test_df.show()

+---------+---------+-----+
|feature_1|feature_2|label|
+---------+---------+-----+
|      0.1|      1.1|  0.0|
|      0.2|      1.3|  0.0|
|      0.3|      1.7|  0.0|
|     0.35|      1.9|  0.0|
|      0.4|      2.2|  0.0|
|     0.55|      2.8|  0.0|
|      0.7|      3.1|  1.0|
|      0.8|      3.3|  1.0|
+---------+---------+-----+

+---------+---------+-----+
|feature_1|feature_2|label|
+---------+---------+-----+
|      0.9|      3.6|  1.0|
|      1.1|      4.0|  1.0|
|     0.25|      1.5|  0.0|
|      1.2|      4.3|  1.0|
+---------+---------+-----+



In [4]:
assembler = VectorAssembler(inputCols=["feature_1", "feature_2"], outputCol="features")
classifier = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    maxIter=30,
    regParam=0.25,
    elasticNetParam=0.1,
)
pipeline = Pipeline(stages=[assembler, classifier])
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy",
)

In [5]:
with mlflow.start_run(run_name="spark-mllib-logistic-regression"):
    model = pipeline.fit(train_df)
    predictions = model.transform(test_df)
    accuracy = evaluator.evaluate(predictions)

    logistic_model = model.stages[-1]

    mlflow.log_params(
        {
            "algorithm": "spark_ml_logistic_regression",
            "train_rows": train_df.count(),
            "test_rows": test_df.count(),
            "max_iter": classifier.getMaxIter(),
            "reg_param": classifier.getRegParam(),
            "elastic_net_param": classifier.getElasticNetParam(),
        }
    )
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("num_features", float(len(logistic_model.coefficients)))
    mlflow.spark.log_model(model, artifact_path="model")

    current_run = mlflow.active_run()
    print(f"Run ID: {current_run.info.run_id}")
    print(f"Accuracy: {accuracy:.4f}")
    predictions.select("feature_1", "feature_2", "label", "prediction", "probability").show(truncate=False)

26/03/26 13:26:35 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/26 13:26:35 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/03/26 13:26:37 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
2026/03/26 13:26:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run ID: 86b60e942f244556a7feafdab1336c3f
Accuracy: 1.0000
+---------+---------+-----+----------+----------------------------------------+
|feature_1|feature_2|label|prediction|probability                             |
+---------+---------+-----+----------+----------------------------------------+
|0.9      |3.6      |1.0  |1.0       |[0.3151772980921632,0.6848227019078368] |
|1.1      |4.0      |1.0  |1.0       |[0.17502357847377192,0.8249764215262281]|
|0.25     |1.5      |0.0  |0.0       |[0.9051470574202213,0.09485294257977872]|
|1.2      |4.3      |1.0  |1.0       |[0.1189921210522386,0.8810078789477614] |
+---------+---------+-----+----------+----------------------------------------+

🏃 View run spark-mllib-logistic-regression at: http://mlflow:5000/#/experiments/1/runs/86b60e942f244556a7feafdab1336c3f
🧪 View experiment at: http://mlflow:5000/#/experiments/1


In [6]:
spark.stop()